# Observations de cétacés MIRACETI : métriques et analyses spatiales
On part du shapefile brut livré par Miraceti (`raw/observations/`), avec le nettoyage repris du premier notebook
(`analyse.ipynb` : dates, taille des groupes, oui/non, réaction au bateau), puis on passe aux **métriques**,
aux **analyses spatiales** et à la **photo-identification**.

La première cellule récupère les données dans le bucket : il faut un fichier `.env` à la racine du dépôt,
voir `docs/miraceti_data/acces.md`. Le fond de carte (`fond_carte.gpkg`, le trait de côte) est facultatif :
sans lui, les cartes s'affichent sans la côte.

Les **sections 11 à 18** ajoutent : contrôle qualité, rythme journalier, comportement, naissances, distance de détection, pêche, part relative des espèces et habitat (MNT du SHOM).


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # .env et data/ sont lus depuis la racine du dépôt
sys.path.insert(0, str(ROOT))
from pipeline.storage import connect  # noqa: E402

bucket = connect()
LIVRAISON = "2026-11-10_shape_obs_miraceti_2014-2026"
bucket.pull(prefix=f"raw/observations/{LIVRAISON}.zip")  # observations brutes
bucket.pull(prefix="curated/observations/")  # fond de carte, s'il est déposé

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter
from shapely.geometry import box

CHEMIN = (
    f"zip://data/raw/observations/{LIVRAISON}.zip"
    "!Shape Obs MIRACETI 2014-2026/MIRACETI_Obs cetaces_2014-2026.shp"
)
FOND = Path("data/curated/observations/fond_carte.gpkg")  # trait de côte, facultatif
TAILLE = 10_000  # taille des mailles, en mètres (10 km)

df = gpd.read_file(CHEMIN)
df = df[df.geometry.notna() & ~df.geometry.is_empty].copy()
print(len(df), "observations")

## Nettoyage
Repris de `analyse.ipynb` : dates, mois, taille du groupe, présence de jeunes, photo-identification,
réaction au bateau.

In [ ]:
# Dates et mois
df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y", errors="coerce")
df["mois"] = df["date"].dt.month

# Taille du groupe : nb_min, sinon nb_estime (valeurs illisibles ou à 0 ignorées)
nb_min = pd.to_numeric(df["nb_min"], errors="coerce").where(lambda s: s > 0)
nb_estime = pd.to_numeric(df["nb_estime"], errors="coerce").where(lambda s: s > 0)
df["taille"] = nb_min.fillna(nb_estime)


def oui_non(colonne):
    texte = colonne.astype(str).str.strip().str.lower()
    return texte.map(
        {
            "oui": "oui",
            "true": "oui",
            "1": "oui",
            "yes": "oui",
            "non": "non",
            "false": "non",
            "0": "non",
            "no": "non",
        }
    )


df["jeunes"] = oui_non(df["pres_jeune"])
df["photo"] = oui_non(df["photo_id"])

# Réaction au bateau : on regroupe les variantes d'écriture
VARIANTES = [
    ("indif", "indifférent"),
    ("attract", "attraction"),
    ("fuite", "fuite / évitement"),
    ("evit", "fuite / évitement"),
    ("approche", "approche"),
    ("etrave", "étrave"),
    ("sonde", "sonde"),
]


def reaction(texte):
    texte = str(texte).lower()
    for debut, nom in VARIANTES:
        if texte.startswith(debut):
            return nom
    return None


df["reaction"] = df["reac_bat"].apply(reaction)

df = df.to_crs("EPSG:2154")  # Lambert 93 : des mètres, pratique pour des mailles
colonnes_origine = [
    c for c in df.columns if c != "geometry"
]  # avant nos colonnes calculées

## Préparation des données
Le shapefile est déjà nettoyé (`mois`, `jeunes`, `taille`, `photo`, `reaction` existent). On ajoute la position en mètres, la date au bon format, le numéro de maille, et le fond de carte.

In [ ]:
df["x_l93"] = df.geometry.x
df["y_l93"] = df.geometry.y
df["obs_id"] = range(len(df))
df["espece"] = df["especes"]
df["date"] = pd.to_datetime(
    df["date"], errors="coerce"
)  # texte AAAA-MM-JJ -> vraie date
df["pres_jeune"] = df["jeunes"]  # déjà "oui" / "non" / vide

# Numéro de maille : on découpe l'espace en carrés de TAILLE mètres
df["maille_x"] = (df["x_l93"] // TAILLE).astype(int)
df["maille_y"] = (df["y_l93"] // TAILLE).astype(int)
print(df.groupby(["maille_x", "maille_y"]).ngroups, "mailles occupées")

In [ ]:
# Zone affichée sur toutes les cartes (en mètres Lambert 93), avec une marge de 20 km
X0, X1 = df["x_l93"].min() - 20_000, df["x_l93"].max() + 20_000
Y0, Y1 = df["y_l93"].min() - 20_000, df["y_l93"].max() + 20_000

cotes = gpd.read_file(FOND) if FOND.exists() else None  # terres émergées, en Lambert 93


def fond(ax):
    """Met en place la zone, les axes en km et le fond de carte."""
    ax.set_xlim(X0, X1)
    ax.set_ylim(Y0, Y1)
    ax.set_aspect("equal")
    ax.grid(False)
    ax.set_xlabel("X Lambert 93 (km)")
    ax.set_ylabel("Y Lambert 93 (km)")
    km = FuncFormatter(lambda v, _: f"{v / 1000:.0f}")
    ax.xaxis.set_major_formatter(km)
    ax.yaxis.set_major_formatter(km)
    if cotes is not None:
        cotes.plot(ax=ax, color="#ece8dc", edgecolor="#9a9a9a", linewidth=0.5, zorder=0)


def carres(tableau):
    """Transforme un tableau de mailles en carrés (GeoDataFrame).

    Pour les dessiner sur le fond de carte.
    """
    formes = [
        box(x * TAILLE, y * TAILLE, (x + 1) * TAILLE, (y + 1) * TAILLE)
        for x, y in zip(tableau["maille_x"], tableau["maille_y"], strict=True)
    ]
    return gpd.GeoDataFrame(tableau, geometry=formes, crs="EPSG:2154")

## 1. Métriques proposées
Les indicateurs de base, calculés avec **une seule fonction** : on peut donc les obtenir pour l'ensemble des données, ou pour n'importe quel sous-ensemble (une espèce, une année, une zone).

In [ ]:
def resume(s):
    """Indicateurs de base pour un ensemble d'observations."""
    jeunes, photo, reaction = (
        s["pres_jeune"].dropna(),
        s["photo"].dropna(),
        s["reaction"].dropna(),
    )
    return pd.Series(
        {
            "observations": len(s),
            "individus (somme des tailles)": s["taille"].sum(),
            "taille médiane du groupe": s["taille"].median(),
            "espèces": s["espece"].nunique(),
            "jours d'observation": s.groupby(["date", "structure"]).ngroups,
            "mailles occupées": s.groupby(["maille_x", "maille_y"]).ngroups,
            "années": s["annee"].nunique(),
            "structures": s["structure"].nunique(),
            "% avec jeunes": (jeunes == "oui").mean() * 100,
            "% photo-identification": (photo == "oui").mean() * 100,
            "% fuite / évitement": (reaction == "fuite / évitement").mean() * 100,
            "mois le plus fréquent": s["mois"].mode().iloc[0],
        }
    )


resume(df).round(0).to_frame("ensemble des données")

In [ ]:
# Par année et par zone : la même fonction, appliquée à chaque sous-ensemble
par_annee = pd.DataFrame({a: resume(s) for a, s in df.groupby("annee")}).T
par_zone = pd.DataFrame({z: resume(s) for z, s in df.groupby("zone")}).T
display(
    par_annee[
        [
            "observations",
            "individus (somme des tailles)",
            "taille médiane du groupe",
            "espèces",
            "jours d'observation",
            "% avec jeunes",
            "% photo-identification",
        ]
    ].round(0)
)
display(
    par_zone[
        [
            "observations",
            "individus (somme des tailles)",
            "taille médiane du groupe",
            "espèces",
            "jours d'observation",
            "% avec jeunes",
            "% photo-identification",
        ]
    ].round(0)
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].bar(par_annee.index.astype(str), par_annee["observations"], color="#0072B2")
axes[0].set_title("Observations par année")
axes[0].tick_params(axis="x", rotation=60)

axes[1].bar(
    par_annee.index.astype(str),
    par_annee["individus (somme des tailles)"],
    color="#009E73",
)
axes[1].set_title("Individus comptés par année (somme des tailles de groupe)")
axes[1].tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

**Ce que ces indicateurs permettent, et ne permettent pas** : ils décrivent ce qui a été *vu*. Pour un **taux de rencontre** (observations par km parcouru ou par heure de sortie), il faut l'effort d'observation, que le shapefile ne contient pas (traces GPS ou durées de sortie) : à demander à Miraceti.

## 2. Où sont les observations ?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6.5))
fond(ax)
couleurs = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7"]
principales = df["espece"].value_counts().index[:5]
for espece, c in zip(principales, couleurs, strict=False):
    s = df[df["espece"] == espece]
    ax.scatter(s["x_l93"], s["y_l93"], s=10, c=c, alpha=0.8, label=espece)
autres = df[~df["espece"].isin(principales)]
ax.scatter(autres["x_l93"], autres["y_l93"], s=10, c="#555555", label="autres")
ax.legend(fontsize=8, loc="upper left")
ax.set_xlabel("X Lambert 93 (km)")
ax.set_ylabel("Y (km)")
ax.set_title("Observations par espèce")
plt.show()

In [ ]:
# Les mêmes observations, en 4 saisons (déc-fév, mar-mai, juin-août, sept-nov)
saison = {
    12: "hiver",
    1: "hiver",
    2: "hiver",
    3: "printemps",
    4: "printemps",
    5: "printemps",
    6: "été",
    7: "été",
    8: "été",
    9: "automne",
    10: "automne",
    11: "automne",
}
df["saison_meteo"] = df["mois"].map(saison)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, nom in zip(axes.ravel(), ["hiver", "printemps", "été", "automne"], strict=True):
    s = df[df["saison_meteo"] == nom]
    fond(ax)
    ax.hexbin(
        s["x_l93"],
        s["y_l93"],
        gridsize=30,
        mincnt=1,
        cmap="magma_r",
        alpha=0.85,
        extent=(X0, X1, Y0, Y1),
    )
    ax.set_title(f"{nom} ({len(s)} obs.)")
plt.tight_layout()
plt.show()

## 3. Régularité : un endroit où l'on a vu des cétacés une fois, ou chaque année ?
On n'a pas les traces des sorties : on ne sait pas où l'on a cherché **sans rien voir**. Ces cartes montrent donc où l'on a *vu* des cétacés.
Pour distinguer un coup de chance d'une zone régulièrement fréquentée, on compte par maille :
le nombre d'observations, de **jours distincts** (une date × une structure) et d'**années distinctes**.

In [ ]:
par_maille = df.groupby(["maille_x", "maille_y"])
mailles = par_maille.agg(
    observations=("obs_id", "size"),
    jours=("date", "nunique"),
    annees=("annee", "nunique"),
    structures=("structure", "nunique"),
    especes=("espece", "nunique"),
).reset_index()
mailles.sort_values("observations", ascending=False).head(10)

In [ ]:
def carte(ax, tableau, colonne, titre, cmap, **options):
    """Un carré coloré par maille, sur le fond de carte."""
    fond(ax)
    carres(tableau).plot(
        column=colonne,
        ax=ax,
        cmap=cmap,
        alpha=0.85,
        legend=True,
        legend_kwds={"shrink": 0.6},
        **options,
    )
    ax.set_title(titre)


fig, axes = plt.subplots(1, 3, figsize=(18, 5))
carte(axes[0], mailles, "observations", "Observations", "magma_r")
carte(axes[1], mailles, "jours", "Jours distincts", "viridis_r")
carte(axes[2], mailles, "annees", "Années distinctes", "plasma_r")
plt.tight_layout()
plt.show()

In [ ]:
# Mailles régulières : observées sur au moins 5 années différentes
regulieres = mailles[mailles["annees"] >= 5].sort_values("annees", ascending=False)
print(
    len(regulieres),
    "mailles observées sur au moins 5 années,",
    regulieres["observations"].sum(),
    "observations",
)
regulieres.head(10)[
    ["maille_x", "maille_y", "observations", "jours", "annees", "structures", "especes"]
]

## 4. Concentration spatiale
Quelle part des observations tient dans les mailles les plus riches ? Plus la courbe est au-dessus de la diagonale, plus les observations sont concentrées.

In [ ]:
tri = (
    mailles["observations"].sort_values(ascending=False).cumsum()
    / mailles["observations"].sum()
    * 100
)
part_mailles = np.arange(1, len(tri) + 1) / len(tri) * 100

plt.figure(figsize=(5.5, 4.5))
plt.plot(part_mailles, tri.values, color="#0072B2", lw=2, label="observations")
plt.plot([0, 100], [0, 100], color="#999999", ls="--", label="répartition égale")
plt.xlabel("% des mailles (les plus riches d'abord)")
plt.ylabel("% cumulé des observations")
plt.title("Concentration des observations")
plt.legend()
plt.show()

dix_pourcent = max(1, len(mailles) // 10)
print(
    f"Les 10 % de mailles les plus riches ({dix_pourcent}) contiennent "
    f"{tri.iloc[dix_pourcent - 1]:.0f} % des observations"
)

**Attention** : une concentration forte peut refléter les **ports de départ** et les zones de sortie habituelles autant que les animaux. Pour trancher, il faut les traces d'effort.

## 5. Zones de présence de jeunes
Part des groupes avec jeunes, par maille, pour les deux dauphins les plus observés (au moins 6 réponses oui/non par maille).

In [ ]:
MIN_REPONSES = 6  # nombre minimal de réponses oui/non par maille
dauphins = df[
    df["espece"].isin(["Grand dauphin", "Dauphin bleu et blanc"])
    & df["pres_jeune"].isin(["oui", "non"])
]
nurserie = dauphins.groupby(["maille_x", "maille_y"]).agg(
    reponses=("pres_jeune", "size"),
    avec_jeunes=("pres_jeune", lambda s: (s == "oui").mean() * 100),
)
nurserie = nurserie[nurserie["reponses"] >= MIN_REPONSES].reset_index()

fig, ax = plt.subplots(figsize=(10, 6.5))
fond(ax)
carres(mailles).plot(ax=ax, color="#bbbbbb", alpha=0.35)
carres(nurserie).plot(
    column="avec_jeunes",
    ax=ax,
    cmap="YlGn",
    vmin=40,
    vmax=100,
    alpha=0.9,
    legend=True,
    legend_kwds={"shrink": 0.6, "label": "% de groupes avec jeunes"},
)
ax.set_title(f"Présence de jeunes ({len(nurserie)} mailles avec assez de réponses)")
plt.show()

## 6. A-t-on changé de zone dans le temps ?
Comparaison de deux périodes : de combien le centre de gravité des observations de chaque espèce s'est-il déplacé ? (espèces avec au moins 10 observations par période)

In [ ]:
df["periode"] = np.where(df["annee"] < 2020, "2013-2019", "2020-2026")
centre = df.groupby(["espece", "periode"])[["x_l93", "y_l93"]].mean().unstack() / 1000
centre["deplacement_km"] = np.hypot(
    centre[("x_l93", "2020-2026")] - centre[("x_l93", "2013-2019")],
    centre[("y_l93", "2020-2026")] - centre[("y_l93", "2013-2019")],
)
effectifs = pd.crosstab(df["espece"], df["periode"]).add_prefix("obs ")
resultat = centre[["deplacement_km"]].droplevel(1, axis=1).join(effectifs)
resultat[resultat[["obs 2013-2019", "obs 2020-2026"]].min(axis=1) >= 10].round(
    0
)  # au moins 10 obs. par période

### Croisement : vrai déplacement des animaux, ou changement d'effort ?
Le centre de gravité bouge si les **animaux** bougent, mais aussi si **ce ne sont plus les mêmes structures** qui observent, ou plus dans les mêmes zones. On croise donc le déplacement avec la structure et la zone d'observation.

In [ ]:
# 1) Qui observe, et où, dans chaque période ?
print("Observations par zone et par période (%) :")
display(pd.crosstab(df["zone"], df["periode"], normalize="columns").mul(100).round(0))

print("Observations par structure et par période :")
display(pd.crosstab(df["structure"], df["periode"]))

In [ ]:
# 2) Le déplacement "à structure constante" :
# on ne compare que les structures présentes dans les deux périodes
lignes = []
for nom, s in df.groupby("structure"):
    avant, apres = s[s["periode"] == "2013-2019"], s[s["periode"] == "2020-2026"]
    if len(avant) >= 10 and len(apres) >= 10:
        km = (
            np.hypot(
                apres["x_l93"].mean() - avant["x_l93"].mean(),
                apres["y_l93"].mean() - avant["y_l93"].mean(),
            )
            / 1000
        )
        lignes.append(
            {
                "structure": nom,
                "obs 2013-2019": len(avant),
                "obs 2020-2026": len(apres),
                "deplacement_km": round(km),
            }
        )
a_structure_constante = pd.DataFrame(lignes)
a_structure_constante

In [ ]:
# 3) Les deux périodes sur une carte
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, periode in zip(axes, ["2013-2019", "2020-2026"], strict=True):
    s = df[df["periode"] == periode]
    fond(ax)
    for zone, c in zip(
        sorted(df["zone"].dropna().unique()),
        ["#0072B2", "#D55E00", "#009E73", "#CC79A7"],
        strict=False,
    ):
        z = s[s["zone"] == zone]
        ax.scatter(z["x_l93"], z["y_l93"], s=10, c=c, alpha=0.8, label=zone)
    ax.set_title(f"{periode} ({len(s)} obs.)")
    ax.legend(fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

**Comment lire** : si le déplacement reste important *à structure constante*, il est plus probablement lié aux animaux ou aux zones de sortie. S'il disparaît, c'est un effet de changement de structures (la période récente est dominée par MIRACETI, absente avant 2021). Dans tous les cas, ce n'est qu'un indice : sans trace d'effort, on ne peut pas conclure.

## 7. Autres croisements
### 7.1 Espèces × saison × zone
Quelles espèces à quelle période, et dans quelle zone ?

In [ ]:
top = df["espece"].value_counts().index[:6]
print("Répartition des espèces par zone (% de chaque espèce) :")
display(
    pd.crosstab(df[df["espece"].isin(top)]["espece"], df["zone"], normalize="index")
    .mul(100)
    .round(0)
)

# Calendrier : part des observations de chaque espèce, mois par mois
calendrier = pd.crosstab(
    df[df["espece"].isin(top)]["espece"], df["mois"], normalize="index"
).mul(100)
fig, ax = plt.subplots(figsize=(10, 3.5))
p = ax.imshow(calendrier.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(12))
ax.set_xticklabels(calendrier.columns)
ax.set_yticks(range(len(calendrier)))
ax.set_yticklabels(calendrier.index)
ax.set_xlabel("mois")
ax.set_title("Quand observe-t-on chaque espèce ? (% des observations de l'espèce)")
plt.colorbar(p, shrink=0.8)
plt.show()

In [ ]:
# Richesse en espèces par maille
# (au moins 5 observations pour que le chiffre ait un sens)
riches = mailles[mailles["observations"] >= 5]
fig, ax = plt.subplots(figsize=(10, 6.5))
carte(
    ax,
    riches,
    "especes",
    "Nombre d'espèces observées par maille (mailles avec au moins 5 observations)",
    "viridis_r",
)
plt.show()

### 7.2 Réaction au bateau × activité humaine
Les animaux fuient-ils plus quand il y a de la pêche, de la plaisance, plusieurs bateaux ? Et où fuient-ils le plus ?

In [ ]:
# Activité humaine : la colonne d'origine contient des variantes d'écriture,
# on regroupe par mots-clés
texte = df["act_hum_as"].astype(str).str.lower()
df["activite"] = np.select(
    [
        texte.str.contains("che"),
        texte.str.contains("plais"),
        texte.str.contains("bateaux"),
        texte.str.contains("whale"),
        texte.str.contains("aucune"),
    ],
    ["pêche", "plaisance", "autres bateaux", "whale watching", "aucune"],
    default="inconnue",
)

# Nombre de bateaux présents
nb = pd.to_numeric(df["nb_bat"], errors="coerce")
df["bateaux"] = pd.cut(nb, [-1, 0, 1, 100], labels=["0", "1", "2 et plus"])

df["fuite"] = df["reaction"] == "fuite / évitement"
avec_reaction = df[df["reaction"].notna()]


def taux_fuite(colonne):
    t = avec_reaction.groupby(colonne, observed=True).agg(
        reponses=("fuite", "size"), fuite_pct=("fuite", lambda s: s.mean() * 100)
    )
    return t[t["reponses"] >= 10].round(0)


print("% de fuite / évitement selon l'activité humaine :")
display(taux_fuite("activite"))
print("% de fuite / évitement selon le nombre de bateaux :")
display(taux_fuite("bateaux"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, colonne in zip(axes, ["activite", "bateaux"], strict=True):
    t = taux_fuite(colonne)
    ax.bar(t.index.astype(str), t["fuite_pct"], color="#D55E00")
    for i, (n, v) in enumerate(zip(t["reponses"], t["fuite_pct"], strict=True)):
        ax.text(i, v + 1, f"n={n}", ha="center", fontsize=8)
    ax.set_ylabel("% fuite / évitement")
    ax.set_title(f"selon : {colonne}")
plt.tight_layout()
plt.show()

In [ ]:
# Où les animaux fuient-ils le plus ? (mailles avec au moins 6 réponses sur la réaction)
fuite = avec_reaction.groupby(["maille_x", "maille_y"]).agg(
    reponses=("fuite", "size"), fuite_pct=("fuite", lambda s: s.mean() * 100)
)
fuite = fuite[fuite["reponses"] >= MIN_REPONSES].reset_index()

fig, ax = plt.subplots(figsize=(10, 6.5))
fond(ax)
carres(mailles).plot(ax=ax, color="#bbbbbb", alpha=0.35)
carres(fuite).plot(
    column="fuite_pct",
    ax=ax,
    cmap="OrRd",
    alpha=0.9,
    legend=True,
    legend_kwds={"shrink": 0.6, "label": "% fuite / évitement"},
)
ax.set_title(f"Réaction au bateau : % de fuite / évitement ({len(fuite)} mailles)")
plt.show()

### 7.3 Jeunes × saison × zone
Les groupes avec jeunes sont-ils plus fréquents à certaines saisons, dans certaines zones ? (dauphins, réponses oui/non)

In [ ]:
d = dauphins.copy()
d["saison_meteo"] = d["mois"].map(saison)
d["avec_jeunes"] = d["pres_jeune"] == "oui"
tableau = (
    d.pivot_table(
        index="zone", columns="saison_meteo", values="avec_jeunes", aggfunc="mean"
    )
    .mul(100)
    .round(0)
)
effectif = d.pivot_table(
    index="zone", columns="saison_meteo", values="avec_jeunes", aggfunc="size"
)
print("% de groupes avec jeunes (dauphins) :")
display(tableau[["hiver", "printemps", "été", "automne"]])
print(
    "Nombre de réponses (à regarder avant de comparer : certaines cases sont petites) :"
)
display(effectif[["hiver", "printemps", "été", "automne"]])

## 8. Une analyse par espèce
Pour chaque espèce observée au moins 40 fois : un tableau comparatif, puis une fiche (carte des mailles, mois, taille des groupes, observations par année).

In [ ]:
ESPECES = (
    df["espece"].value_counts().loc[lambda s: s >= 40].index
)  # espèces avec au moins 40 observations

pd.DataFrame({e: resume(df[df["espece"] == e]) for e in ESPECES}).T.round(0)

In [ ]:
for nom in ESPECES:
    s = df[df["espece"] == nom]
    par_maille_espece = (
        s.groupby(["maille_x", "maille_y"]).size().rename("observations").reset_index()
    )
    zones = s["zone"].value_counts(normalize=True).mul(100).round(0)

    fig, axes = plt.subplots(
        1, 4, figsize=(22, 5), gridspec_kw={"width_ratios": [2.3, 1, 1, 1]}
    )
    # 1) carte : nombre d'observations par maille
    fond(axes[0])
    carres(mailles).plot(ax=axes[0], color="#bbbbbb", alpha=0.3)
    carres(par_maille_espece).plot(
        column="observations",
        ax=axes[0],
        cmap="magma_r",
        alpha=0.9,
        legend=True,
        legend_kwds={"shrink": 0.6},
    )
    axes[0].set_title(f"{nom} : {len(s)} observations")
    # 2) calendrier
    par_mois = s["mois"].value_counts().reindex(range(1, 13), fill_value=0)
    axes[1].bar(par_mois.index, par_mois.values, color="#0072B2")
    axes[1].set_xticks(range(1, 13))
    axes[1].set_title("Par mois")
    # 3) taille des groupes (au-delà de 30 individus, on regroupe)
    axes[2].hist(
        s["taille"].dropna().clip(upper=30),
        bins=range(0, 32, 2),
        color="#E69F00",
        edgecolor="white",
    )
    axes[2].set_title("Taille des groupes (30 = 30 et plus)")
    # 4) par année
    par_annee = s["annee"].value_counts().sort_index()
    axes[3].bar(par_annee.index, par_annee.values, color="#009E73")
    axes[3].set_title("Par année")
    axes[3].tick_params(axis="x", rotation=60)
    plt.tight_layout()
    plt.show()

    print("Zones (%) :", ", ".join(f"{z} {v:.0f}" for z, v in zones.items()))

**À garder en tête** : le nombre d'observations par mois ou par année suit l'effort d'observation (les sorties se font surtout l'été, et MIRACETI domine depuis 2021). Une espèce n'est donc pas forcément « plus présente » en septembre : on l'y voit plus souvent.

## 9. Photo-identification
Pour les analyses de photo-identification et d'apprentissage automatique, on garde les observations **avec photo** (`photo == "oui"`).
Attention : le fichier `photo_tous_attributs.shp` exporté dans `analyse.ipynb` contient toutes les lignes où la colonne `photo` est renseignée, y compris les `non`. Ici on repart du fichier nettoyé et on filtre sur `oui`.

In [ ]:
reponses = df[df["photo"].notna()]  # observations où la question photo a été renseignée
print(
    len(reponses),
    "réponses,",
    (reponses["photo"] == "oui").sum(),
    "observations avec photo",
)


def part_photo(colonne, minimum=10):
    t = reponses.groupby(colonne).agg(
        reponses=("photo", "size"), avec_photo=("photo", lambda s: (s == "oui").sum())
    )
    t["% avec photo"] = (t["avec_photo"] / t["reponses"] * 100).round(0)
    return t[t["reponses"] >= minimum].sort_values("% avec photo", ascending=False)


display(part_photo("espece"))
display(part_photo("structure"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
t = part_photo("structure")
axes[0].barh(t.index.str[:30], t["% avec photo"], color="#0072B2")
axes[0].set_title("% d'observations avec photo, par structure")
axes[0].invert_yaxis()

t = part_photo("annee", minimum=5).sort_index()
axes[1].bar(t.index.astype(str), t["% avec photo"], color="#009E73")
axes[1].set_title("% d'observations avec photo, par année")
axes[1].tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.show()

Le taux de photo-identification dépend beaucoup de l'**espèce** et de la **structure** : le Dauphin bleu et blanc n'est presque jamais photographié, alors que le Grand dauphin l'est dans la grande majorité des cas. Il faut donc comparer ce qui est comparable (même espèce, même structure).

In [ ]:
# Où a-t-on photographié ? (observations avec et sans photo)
fig, ax = plt.subplots(figsize=(10, 6.5))
fond(ax)
sans = df[df["photo"] == "non"]
avec = df[df["photo"] == "oui"]
ax.scatter(
    sans["x_l93"],
    sans["y_l93"],
    s=8,
    c="#999999",
    alpha=0.6,
    label=f"sans photo ({len(sans)})",
)
ax.scatter(
    avec["x_l93"],
    avec["y_l93"],
    s=10,
    c="#D55E00",
    alpha=0.8,
    label=f"avec photo ({len(avec)})",
)
ax.legend(loc="upper left")
ax.set_title("Observations avec et sans photo")
plt.show()

### Potentiel pour la recapture photo (Grand dauphin)
Pour suivre des individus dans le temps, il faut des zones photographiées **sur plusieurs années**. Pour le Grand dauphin, on compte par maille le nombre de photos, de jours et d'années distinctes.

In [ ]:
gd = df[(df["espece"] == "Grand dauphin") & (df["photo"] == "oui")]
photo_mailles = (
    gd.groupby(["maille_x", "maille_y"])
    .agg(
        observations=("obs_id", "size"),
        jours=("date", "nunique"),
        annees=("annee", "nunique"),
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 6.5))
carte(
    ax,
    photo_mailles,
    "annees",
    "Grand dauphin avec photo : nombre d'années distinctes par maille",
    "viridis_r",
)
plt.show()

recapture = photo_mailles[photo_mailles["annees"] >= 3]
print(
    len(recapture),
    "mailles photographiées sur au moins 3 années,",
    recapture["observations"].sum(),
    "observations",
)
recapture.sort_values("annees", ascending=False).head(10)

### Jeu de données pour le machine learning
Quelles colonnes sont bien remplies **parmi les observations avec photo** ? Ce sont celles qu'on peut utiliser directement comme variables. Le fichier `photo_attributs_complets.shp` (colonnes remplies à 100 %) ne contient que des champs d'identification (date, lieu, structure, espèce) : un seuil à 90 % donne plus de variables utiles.

In [ ]:
avec_photo = df.loc[df["photo"] == "oui", colonnes_origine]
completude = (
    avec_photo.replace(r"^\s*$", pd.NA, regex=True).notna().mean().mul(100).round(0)
)
utiles = completude[completude >= 90].sort_values(ascending=False)
print(
    len(avec_photo),
    "observations avec photo,",
    len(utiles),
    "colonnes remplies à 90 % ou plus",
)
utiles.to_frame("% rempli")

**Question pour Miraceti** : la colonne `repres` (renseignée seulement pour une minorité d'observations) semble décrire la qualité ou la représentativité de la photo-identification ; à confirmer avant de s'en servir pour filtrer un jeu d'entraînement.

## 10. Une carte publique sans exposer les positions
Pour une carte en ligne, on n'affiche pas les points mais des **mailles avec un seuil minimal** d'observations.

In [ ]:
SEUIL = 5
publiques = mailles[mailles["observations"] >= SEUIL]
masquees = len(df) - publiques["observations"].sum()
print(
    f"Mailles de {TAILLE // 1000} km, seuil {SEUIL} observations : "
    f"{len(publiques)} mailles publiables, "
    f"{masquees} observations masquées ({100 * masquees / len(df):.0f} %)"
)

fig, ax = plt.subplots(figsize=(10, 6.5))
carte(
    ax,
    publiques,
    "observations",
    "Version publique : mailles avec au moins 5 observations",
    "magma_r",
    vmax=60,
)
plt.show()

In [ ]:
# Export pour QGIS : les mailles publiables en GeoPackage (sans les positions exactes)
# Écrit dans data/, jamais dans Git
SORTIE = Path("data/derived/observations/mailles_publiques.gpkg")
SORTIE.parent.mkdir(parents=True, exist_ok=True)
colonnes = ["maille_x", "maille_y", "observations", "jours", "annees"]
carres(publiques[colonnes]).to_file(SORTIE, driver="GPKG")
print("Écrit :", SORTIE)

# Analyses complémentaires
Suite du notebook, sur les mêmes données (`df`) et avec les mêmes outils (`fond`, `carres`, `carte`, `mailles`). Sections 11 à 18 :
nettoyage et contrôle qualité, rythme journalier, comportement, naissances, distance de détection, pêche, part relative des espèces, et **habitat** (bathymétrie du SHOM).

Pour la section 18 (habitat), il faut les **MNT du SHOM** (100 m, WGS84) : la liste `MNT` est au début de cette section, à adapter si les fichiers ne sont pas dans `data/raw/mnt_shom/`.

## 11. Nettoyage et contrôle qualité
Toutes les étapes sont regroupées dans **une seule cellule**, qui les **applique** dans l'ordre :
1. fond de carte fusionné en une géométrie (`terre`) ;
2. colonnes ajoutées pour la suite : heure décimale, comportement, nouveau-nés, distance de détection, engin de pêche ;
3. contrôles de cohérence : dates, heures, tailles de groupe, écart entre la géométrie et `latitude` / `longitude` ;
4. **suppression des doublons UTC / heure locale** : même jour, même position, même espèce et même taille de groupe, avec des heures qui diffèrent d'**exactement 1 h (hiver) ou 2 h (été)**. On garde la ligne en heure locale (la plus tardive) ;
5. **suppression des points à plus de 1 km à l'intérieur des terres** (les points à moins de 1 km sont gardés : marge du trait de côte).

Les lignes retirées sont conservées dans `retirees` pour pouvoir les transmettre à Miraceti.

In [ ]:
from shapely.geometry import LineString, MultiLineString

# 1) Trait de côte fusionné en une seule géométrie (None si le fond de carte est absent)
terre = None
if cotes is not None:
    terre = cotes.geometry.union_all() if hasattr(cotes.geometry, "union_all") else cotes.geometry.unary_union

# 2) Colonnes supplémentaires, à partir des colonnes d'origine du shapefile
heure = pd.to_datetime(df["heure"], format="%H:%M:%S.%f", errors="coerce")
df["heure_h"] = heure.dt.hour + heure.dt.minute / 60  # 13h30 -> 13.5


def comportement(texte):
    """Comportement au début de l'observation : on garde le premier cité et on regroupe les variantes."""
    t = str(texte).split(";")[0].strip().lower()
    for mot, nom in [("alimen", "alimentation"), ("chasse", "alimentation"), ("repos", "repos"),
                     ("social", "activités sociales"), ("deplacement", "déplacement"),
                     ("voyage", "déplacement"), ("indetermin", "indéterminé")]:
        if mot in t:
            return nom
    return None if t in ("", "nan", "na") else "autre"


def engin(texte):
    """Engin de pêche présent, regroupé."""
    t = str(texte).strip().lower()
    for mot, nom in [("chalut", "chalut"), ("filet", "filet maillant"), ("palangre", "palangre"),
                     ("casier", "casiers"), ("aucune", "aucune"), ("autre", "autre")]:
        if t.startswith(mot):
            return nom
    return None


df["comportement"] = df["cpt_debut"].apply(comportement)
df["nouveau_ne"] = oui_non(df["pres_nouv"])
df["distance_m"] = pd.to_numeric(df["distance"], errors="coerce")  # distance au groupe, en mètres
df["engin_peche"] = df["act_peche"].apply(engin)
df["date_heure_locale"] = pd.to_datetime(df["date_heure"], format="%d/%m/%Y %H:%M", errors="coerce")

# 3) Contrôles de cohérence (avant nettoyage)
wgs = df.to_crs("EPSG:4326")
ecart_km = np.hypot((df["longitude"] - wgs.geometry.x) * 111.32 * np.cos(np.radians(wgs.geometry.y)),
                    (df["latitude"] - wgs.geometry.y) * 111.2)
controle = pd.Series({
    "observations au départ": len(df),
    "date manquante": df["date"].isna().sum(),
    "date avant 2013 ou dans le futur": ((df["date"] < "2013-01-01") | (df["date"] > pd.Timestamp.today())).sum(),
    "heure manquante": df["heure_h"].isna().sum(),
    "heure avant 5h ou après 21h": ((df["heure_h"] < 5) | (df["heure_h"] > 21)).sum(),
    "taille de groupe manquante": df["taille"].isna().sum(),
    "taille de groupe de plus de 100": (df["taille"] > 100).sum(),
    "géométrie à plus de 1 km de latitude / longitude": int((ecart_km > 1).sum()),
})

# 4) Doublons UTC / heure locale : mêmes jour, position, espèce, taille, heures écartées de 1 h ou 2 h
cle = ["date", "x_l93", "y_l93", "espece", "taille"]
groupe = df.groupby(cle, dropna=False)["date_heure_locale"]
ecart_heures = groupe.transform("max") - groupe.transform("min")
doublon_utc = ecart_heures.isin([pd.Timedelta(hours=1), pd.Timedelta(hours=2)]) & (df["date_heure_locale"] < groupe.transform("max"))
autres_doublons = df.duplicated(cle, keep=False) & ~ecart_heures.isin([pd.Timedelta(0), pd.Timedelta(hours=1), pd.Timedelta(hours=2)]) & ecart_heures.notna()

# 5) Points à plus de 1 km à l'intérieur des terres
if terre is not None:
    a_terre = df.geometry.within(terre)
    dans_terres = a_terre & (df.geometry.distance(terre.boundary) > 1000)
else:
    a_terre = dans_terres = pd.Series(False, index=df.index)
    print("Fond de carte absent : test des points à terre ignoré")

controle["doublons UTC / heure locale (supprimés)"] = int(doublon_utc.sum())
controle["autres doublons (gardés, à vérifier)"] = int(autres_doublons.sum())
controle["points sur la terre (à moins de 1 km : gardés)"] = int(a_terre.sum() - dans_terres.sum())
controle["points à plus de 1 km dans les terres (supprimés)"] = int(dans_terres.sum())

# Application
retirees = df[doublon_utc | dans_terres].copy()
df = df[~(doublon_utc | dans_terres)].copy()
controle["observations restantes"] = len(df)
print(len(retirees), "lignes retirées ;", len(df), "observations conservées")
print("Espèces analysées :", ", ".join(ESPECES))

In [ ]:
# Bilan du nettoyage, lignes retirées, et plus grands groupes à vérifier
display(controle.to_frame("lignes concernées"))
display(retirees[["obs_id", "date_heure", "structure", "espece", "taille", "x_l93", "y_l93"]].sort_values("date_heure"))
display(df.nlargest(10, "taille")[["obs_id", "date", "structure", "espece", "taille"]])

fig, ax = plt.subplots(figsize=(10, 6.5))
fond(ax)
ax.scatter(df["x_l93"], df["y_l93"], s=5, c="#bbbbbb", label=f"conservées ({len(df)})")
ax.scatter(retirees["x_l93"], retirees["y_l93"], s=40, c="#D55E00", label=f"retirées ({len(retirees)})")
ax.legend(loc="upper left")
ax.set_title("Nettoyage : observations retirées (doublons UTC et points dans les terres)")
plt.show()

## 12. Rythme journalier
À quelle heure observe-t-on chaque espèce ? Les sorties se font de jour, donc la courbe suit surtout **l'heure des sorties** ; on cherche des écarts entre espèces.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
df["heure_h"].dropna().plot.hist(bins=range(5, 22), ax=axes[0], color="#0072B2", edgecolor="white")
axes[0].set_title("Observations par heure (toutes espèces)")
axes[0].set_xlabel("heure")

for espece, couleur in zip(ESPECES[:5], ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7"]):
    heures = df.loc[df["espece"] == espece, "heure_h"].dropna()
    axes[1].hist(heures, bins=range(5, 22, 2), density=True, histtype="step", lw=2, color=couleur, label=f"{espece} (n={len(heures)})")
axes[1].set_title("Répartition dans la journée, par espèce (proportion)")
axes[1].set_xlabel("heure")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

df[df["espece"].isin(ESPECES)].groupby("espece")["heure_h"].describe()[["count", "25%", "50%", "75%"]].round(1)

## 13. Comportement
Comportement noté au **début** de l'observation (`cpt_debut`, regroupé : alimentation, repos, activités sociales, déplacement, indéterminé). Quand plusieurs comportements sont cités, on garde le premier.

In [ ]:
comp = df[df["espece"].isin(ESPECES) & df["comportement"].notna()]
tableau_comp = pd.crosstab(comp["comportement"], comp["espece"], normalize="columns").mul(100).round(0)
effectifs_comp = comp["espece"].value_counts()
print("% de chaque espèce dans chaque comportement (réponses :", ", ".join(f"{e} {n}" for e, n in effectifs_comp.items()), ")")
display(tableau_comp)

tableau_comp.T.plot.barh(stacked=True, figsize=(10, 3.8), colormap="Set2")
plt.xlabel("% des observations de l'espèce")
plt.title("Comportement au début de l'observation")
plt.legend(fontsize=8, bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

In [ ]:
# Comportement et réaction au bateau : fuit-on plus quand on se nourrit, quand on se repose ?
avec_cpt = df[df["comportement"].notna() & df["reaction"].notna()]
t = avec_cpt.groupby("comportement").agg(reponses=("fuite", "size"), fuite_pct=("fuite", lambda s: s.mean() * 100)).round(0)
display(t[t["reponses"] >= 15])

# Comportements de surface notés dans `autres_cpt` (sauts, étrave...) : part des observations concernées, par espèce
texte_cpt = df["autres_cpt"].astype(str).str.lower()
df["saut"] = texte_cpt.str.contains("saut|chandelle")
df["etrave"] = texte_cpt.str.contains("etrave|étrave")
sous = df[df["espece"].isin(ESPECES) & df["autres_cpt"].notna() & (texte_cpt != "na")]
sous.groupby("espece").agg(reponses=("saut", "size"), saut_pct=("saut", lambda s: s.mean() * 100),
                           etrave_pct=("etrave", lambda s: s.mean() * 100)).round(0)

## 14. Naissances
`pres_nouv` : présence de **nouveau-nés** dans le groupe. C'est plus précis que la simple présence de jeunes, presque toujours positive. Les « ne sais pas » sont exclus.

In [ ]:
dauphins_nn = df[df["espece"].isin(["Grand dauphin", "Dauphin bleu et blanc"]) & df["nouveau_ne"].notna()].copy()
dauphins_nn["avec_nouveau_ne"] = (dauphins_nn["nouveau_ne"] == "oui")
print(len(dauphins_nn), "observations de dauphins avec réponse ;", f"{dauphins_nn['avec_nouveau_ne'].mean() * 100:.0f} % avec nouveau-nés")

nn_mois = dauphins_nn.groupby(["espece", "mois"]).agg(reponses=("avec_nouveau_ne", "size"), pct=("avec_nouveau_ne", lambda s: s.mean() * 100)).reset_index()
nn_mois = nn_mois[nn_mois["reponses"] >= 10]

fig, ax = plt.subplots(figsize=(9, 3.8))
for (espece, groupe), couleur in zip(nn_mois.groupby("espece"), ["#0072B2", "#E69F00"]):
    ax.plot(groupe["mois"], groupe["pct"], marker="o", color=couleur, label=espece)
ax.set_xticks(range(1, 13))
ax.set_xlabel("mois")
ax.set_ylabel("% de groupes avec nouveau-nés")
ax.set_title("Quand voit-on des nouveau-nés ? (mois avec au moins 10 réponses)")
ax.legend()
plt.show()

In [ ]:
# Par zone et saison (les petits effectifs sont à lire avec prudence)
tab = dauphins_nn.pivot_table(index="zone", columns="saison_meteo", values="avec_nouveau_ne", aggfunc="mean").mul(100).round(0)
eff = dauphins_nn.pivot_table(index="zone", columns="saison_meteo", values="avec_nouveau_ne", aggfunc="size")
ordre_saisons = [s for s in ["hiver", "printemps", "été", "automne"] if s in tab.columns]
print("% de groupes avec nouveau-nés :")
display(tab[ordre_saisons])
print("Nombre de réponses :")
display(eff[ordre_saisons])

## 15. Distance de détection
`distance` : distance (en mètres) entre le bateau et le groupe au début de l'observation, estimée à l'œil. Elle dit à quelle distance on repère chaque espèce.
Pour en tirer une **probabilité de détection** (distance sampling), il faudrait les lignes d'effort.

In [ ]:
det = df[df["espece"].isin(ESPECES) & df["distance_m"].between(1, 5000)]
ordre_det = det.groupby("espece")["distance_m"].median().sort_values().index
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.boxplot([det.loc[det["espece"] == e, "distance_m"] for e in ordre_det], showfliers=False)
ax.set_xticks(range(1, len(ordre_det) + 1))
ax.set_xticklabels(ordre_det, rotation=20, ha="right")
ax.set_ylabel("distance au groupe (m)")
ax.set_title("Distance de détection par espèce")
plt.tight_layout()
plt.show()

det.groupby("espece")["distance_m"].describe()[["count", "25%", "50%", "75%"]].round(0)

In [ ]:
# Les gros groupes se repèrent-ils de plus loin ?
classes_taille = pd.cut(df["taille"], [0, 1, 5, 10, 20, 1000], labels=["1", "2-5", "6-10", "11-20", "plus de 20"])
valide = df["distance_m"].between(1, 5000)
df[valide].groupby(classes_taille[valide], observed=True)["distance_m"].agg(["count", "median"]).round(0)

## 16. Pêche
Quels engins de pêche croise-t-on, avec quelles espèces, dans quelles zones ? (`act_peche`, regroupé ; seulement les observations où la question est renseignée.)

In [ ]:
peche = df[df["engin_peche"].notna()]
print(len(peche), "observations avec réponse sur l'engin de pêche")
display(pd.crosstab(peche["engin_peche"], peche["zone"], margins=True))
display(pd.crosstab(peche[peche["espece"].isin(ESPECES)]["engin_peche"], peche[peche["espece"].isin(ESPECES)]["espece"]))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6.5))
fond(ax)
ax.scatter(df["x_l93"], df["y_l93"], s=5, c="#cccccc", label="toutes les observations")
for nom_engin, couleur in [("chalut", "#D55E00"), ("filet maillant", "#0072B2")]:
    s = peche[peche["engin_peche"] == nom_engin]
    ax.scatter(s["x_l93"], s["y_l93"], s=22, c=couleur, label=f"{nom_engin} ({len(s)})")
ax.legend(loc="upper left")
ax.set_title("Observations avec un chalut ou un filet maillant à proximité")
plt.show()

# Réaction au bateau selon l'engin
t = peche[peche["reaction"].notna()].groupby("engin_peche").agg(reponses=("fuite", "size"), fuite_pct=("fuite", lambda s: s.mean() * 100)).round(0)
t[t["reponses"] >= 10]

**Attention** : la question sur les engins de pêche n'est renseignée que par certaines structures (presque tous les chaluts viennent du Golfe du Lion). La carte montre donc autant **où l'on a posé la question** que où l'on pêche : à ne pas lire comme une carte de la pêche.

## 17. Part relative des espèces : qui domine où ?
Une carte d'observations d'une espèce dépend de l'effort. Pour en limiter l'effet, on regarde, dans chaque maille, la **part de l'espèce parmi toutes les observations** et on la compare à sa part sur l'ensemble de la zone. Les deux sont vues avec le même effort, donc le rapport est plus fiable.
Rouge : l'espèce est **sur-représentée** dans la maille. Bleu : **sous-représentée**. Mailles avec au moins 10 observations.

In [ ]:
MIN_OBS = 10
total_maille = df.groupby(["maille_x", "maille_y"]).size().rename("total")

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, espece in zip(axes.ravel(), ESPECES[:4]):
    n = df[df["espece"] == espece].groupby(["maille_x", "maille_y"]).size().rename("n")
    t = pd.concat([total_maille, n], axis=1).fillna(0).reset_index()
    t = t[t["total"] >= MIN_OBS].copy()
    part_globale = (df["espece"] == espece).mean()
    # log2 du rapport : 0 = comme la moyenne, +1 = deux fois plus, -1 = deux fois moins (le +0,5 évite les zéros)
    t["rapport"] = np.log2((t["n"] + 0.5) / (t["total"] * part_globale + 0.5))
    fond(ax)
    carres(t).plot(column="rapport", ax=ax, cmap="RdBu_r", vmin=-3, vmax=3, alpha=0.9, legend=True,
                   legend_kwds={"shrink": 0.6, "label": "log2 (part locale / part globale)"})
    ax.set_title(f"{espece} (part globale {part_globale * 100:.0f} %, {len(t)} mailles)")
plt.tight_layout()
plt.show()

In [ ]:
# Les deux dauphins l'un contre l'autre : part du Grand dauphin parmi les deux
deux = df[df["espece"].isin(["Grand dauphin", "Dauphin bleu et blanc"])]
t = deux.groupby(["maille_x", "maille_y"]).agg(total=("obs_id", "size"), grand_dauphin=("espece", lambda s: (s == "Grand dauphin").mean() * 100)).reset_index()
t = t[t["total"] >= MIN_OBS]

fig, ax = plt.subplots(figsize=(10, 6.5))
fond(ax)
carres(t).plot(column="grand_dauphin", ax=ax, cmap="PuOr", vmin=0, vmax=100, alpha=0.9, legend=True,
               legend_kwds={"shrink": 0.6, "label": "% de Grand dauphin (0 = Dauphin bleu et blanc)"})
ax.set_title(f"Grand dauphin ou Dauphin bleu et blanc ? ({len(t)} mailles avec au moins {MIN_OBS} observations des deux)")
plt.show()

## 18. Habitat : bathymétrie du SHOM
On croise les observations avec le **MNT du SHOM** (Golfe du Lion et Corse : `MNT_MED100m_..._HOMONIM_WGS84_NM_ZNEG`, 100 m, profondeurs négatives, référencées au niveau moyen). Pour chaque observation : profondeur, pente, distance à la côte, distance au talus (isobathe de 200 m).

Les profondeurs sont au **niveau moyen** (NM), pas au zéro hydrographique : l'écart (de l'ordre de quelques dizaines de centimètres en Méditerranée) est négligeable pour des classes de plusieurs dizaines de mètres.

Le biais d'effort joue ici aussi : on observe surtout près des côtes. Les comparaisons se font donc **entre espèces** et par rapport à **l'ensemble des observations**, jamais par rapport à la mer entière.

In [ ]:
MNT = [  # MNT du SHOM, un par façade (les .grd conviennent aussi ; le .glz n'est pas lu)
    Path("data/raw/mnt_shom/MNT_MED100m_GDL-CA_HOMONIM_WGS84_NM_ZNEG.asc"),
    Path("data/raw/mnt_shom/MNT_MED100m_CORSE_HOMONIM_WGS84_NM_ZNEG.asc"),
]
RESOLUTION = 250  # taille des pixels de la grille de bathymétrie de travail, en mètres
# Si les MNT sont dans le bucket, les récupérer d'abord (adapter le préfixe) :
# bucket.pull(prefix="raw/mnt_shom/")

# Vérifications avant chargement
try:
    import rasterio
except ImportError as erreur:
    raise ImportError("rasterio est nécessaire pour lire les MNT (pip install rasterio, ou l'ajouter aux dépendances du projet)") from erreur

with rasterio.Env() as env:
    pilotes = set(env.drivers())
print("rasterio", rasterio.__version__, "| pilotes :", {p: (p in pilotes) for p in ["AAIGrid", "netCDF", "BAG"]})
EXTENSION_PILOTE = {".asc": "AAIGrid", ".grd": "netCDF", ".nc": "netCDF", ".bag": "BAG", ".tif": "GTiff"}

lignes = []
etendue = []
for chemin in MNT:
    chemin = Path(chemin)
    ligne_verif = {"fichier": chemin.name, "existe": chemin.exists(), "taille_Mo": None, "pilote": EXTENSION_PILOTE.get(chemin.suffix.lower(), "non pris en charge"),
                   "lisible": False, "pixels": None, "projection": None, "nodata": None, "memoire_Mo": None}
    if chemin.exists():
        ligne_verif["taille_Mo"] = round(chemin.stat().st_size / 1e6)
        try:
            with rasterio.open(chemin) as source:
                ligne_verif.update(lisible=True, pixels=f"{source.width} x {source.height}", projection=str(source.crs or "aucune (WGS84 supposé)"),
                                   nodata=source.nodata, memoire_Mo=round(source.width * source.height * 4 / 1e6))
                if source.crs is None or source.crs.is_geographic:
                    etendue.append(source.bounds)  # en degrés WGS84
        except Exception as erreur:
            ligne_verif["lisible"] = f"ERREUR : {erreur}"
    lignes.append(ligne_verif)
display(pd.DataFrame(lignes).set_index("fichier"))

probleme = [l["fichier"] for l in lignes if not l["existe"] or l["lisible"] is not True]
if probleme:
    raise FileNotFoundError("MNT absent(s) ou illisible(s) : " + ", ".join(probleme) + "  -> corriger la liste MNT ci-dessus (ou utiliser les .grd si le .asc pose problème)")

# Les observations sont-elles couvertes par les MNT ? (comparaison en degrés WGS84)
wgs_obs = df.to_crs("EPSG:4326")
couvertes = np.zeros(len(df), dtype=bool)
for b in etendue:
    couvertes |= ((wgs_obs.geometry.x >= b.left) & (wgs_obs.geometry.x <= b.right) & (wgs_obs.geometry.y >= b.bottom) & (wgs_obs.geometry.y <= b.top)).to_numpy()
print(int(couvertes.sum()), "observations sur", len(df), "dans l'emprise des MNT ;", int((~couvertes).sum()), "en dehors (elles n'auront pas de profondeur)")

In [ ]:
from rasterio.transform import from_origin
from rasterio.warp import Resampling, reproject

manquants = [str(m) for m in MNT if not Path(m).exists()]
if manquants:
    raise FileNotFoundError("MNT introuvable(s) : " + " ; ".join(manquants) + "  -> corriger la liste MNT plus haut")

# Grille de travail : tous les MNT sont rééchantillonnés (moyenne) sur une même grille Lambert 93 de RESOLUTION mètres
largeur, hauteur = int((X1 - X0) // RESOLUTION) + 1, int((Y1 - Y0) // RESOLUTION) + 1
transformation = from_origin(X0, Y1, RESOLUTION, RESOLUTION)
z = np.full((hauteur, largeur), np.nan, dtype="float32")
for chemin in MNT:
    with rasterio.open(chemin) as source:
        projection = source.crs or "EPSG:4326"  # les .asc et .grd du SHOM n'ont pas de projection : WGS84 (degrés), comme l'indique leur nom
        print(Path(chemin).name, "|", source.crs or "pas de projection -> WGS84 supposé", "|", source.res, "| nodata :", source.nodata)
        donnees = source.read(1).astype("float32")
        if source.nodata is not None and not np.isnan(source.nodata):
            donnees[donnees == source.nodata] = np.nan
        reproject(source=donnees, src_transform=source.transform, src_crs=projection, destination=z,
                  dst_transform=transformation, dst_crs="EPSG:2154", src_nodata=np.nan, dst_nodata=np.nan,
                  resampling=Resampling.average, init_dest_nodata=False)
z[np.abs(z) > 20_000] = np.nan  # valeurs "pas de données" non déclarées (-9999, 99999...)

# Convention de signe : profondeurs négatives (cas le plus courant) ou positives ?
negatif = np.nanmedian(z) < 0
profondeur = -z if negatif else z.copy()
profondeur[profondeur < 0] = np.nan  # terres émergées (MNT topo-bathy) : pas de profondeur
print("Profondeurs lues comme", "négatives" if negatif else "positives", "dans le MNT ; profondeur médiane de la grille :", int(np.nanmedian(profondeur)), "m")

In [ ]:
# Pente (degrés) à l'échelle de la grille
gy, gx = np.gradient(profondeur.astype("float64"), RESOLUTION)
pente = np.degrees(np.arctan(np.hypot(gx, gy)))

# Isobathe 200 m (talus) avec matplotlib : on retourne la grille pour que y soit croissant
xs = X0 + (np.arange(largeur) + 0.5) * RESOLUTION
ys = Y1 - (hauteur - 1 - np.arange(hauteur) + 0.5) * RESOLUTION
fig_tmp, ax_tmp = plt.subplots()
cs = ax_tmp.contour(xs, ys, np.ma.masked_invalid(profondeur[::-1]), levels=[200])
troncons = [LineString(p) for chemin in cs.get_paths() for p in chemin.to_polygons(closed_only=False) if len(p) > 1]
plt.close(fig_tmp)
talus = MultiLineString(troncons) if troncons else None
print(len(troncons), "tronçons d'isobathe de 200 m")

# Valeurs aux points d'observation
colonne = ((df.geometry.x - X0) // RESOLUTION).astype(int)
ligne = ((Y1 - df.geometry.y) // RESOLUTION).astype(int)
dedans = (colonne >= 0) & (colonne < largeur) & (ligne >= 0) & (ligne < hauteur)
c, l = colonne.clip(0, largeur - 1), ligne.clip(0, hauteur - 1)
df["profondeur"] = np.where(dedans, profondeur[l, c], np.nan).astype(float)
df["pente"] = np.where(dedans, pente[l, c], np.nan).astype(float)

# Distance à la côte et au talus
df["dist_cote_km"] = df.geometry.distance(terre.boundary) / 1000 if terre is not None else np.nan
df["dist_talus_km"] = df.geometry.distance(talus) / 1000 if talus is not None else np.nan

print(df["profondeur"].notna().sum(), "observations sur", len(df), "avec une profondeur (les autres sont hors des MNT ou à terre)")
df[["profondeur", "pente", "dist_cote_km", "dist_talus_km"]].describe().round(1)

In [ ]:
# Carte : bathymétrie, isobathes 200 m et 1000 m, observations des 4 espèces principales
fig, ax = plt.subplots(figsize=(11, 7))
image = ax.imshow(profondeur, extent=(X0, X1, Y0, Y1), cmap="Blues", vmin=0, vmax=3000, zorder=0)
plt.colorbar(image, ax=ax, shrink=0.6, label="profondeur (m)")
ax.contour(xs, ys, profondeur[::-1], levels=[200, 1000], colors=["#555555", "#222222"], linewidths=0.5, zorder=1)
if cotes is not None:
    cotes.plot(ax=ax, color="#ece8dc", edgecolor="#9a9a9a", linewidth=0.5, zorder=2)
for espece, couleur in zip(ESPECES[:4], ["#E69F00", "#D55E00", "#009E73", "#CC79A7"]):
    s = df[df["espece"] == espece]
    ax.scatter(s["x_l93"], s["y_l93"], s=7, c=couleur, label=espece, zorder=3)
ax.set_xlim(X0, X1)
ax.set_ylim(Y0, Y1)
ax.set_aspect("equal")
ax.legend(loc="upper left", fontsize=8)
km = FuncFormatter(lambda v, _: f"{v / 1000:.0f}")
ax.xaxis.set_major_formatter(km)
ax.yaxis.set_major_formatter(km)
ax.set_xlabel("X Lambert 93 (km)")
ax.set_ylabel("Y Lambert 93 (km)")
ax.set_title("Observations sur la bathymétrie du SHOM")
plt.show()

In [ ]:
# Habitat de chaque espèce : profondeur, pente, distance à la côte, distance au talus
variables = [("profondeur", "profondeur (m)", True), ("pente", "pente (°)", False),
             ("dist_cote_km", "distance à la côte (km)", False), ("dist_talus_km", "distance au talus 200 m (km)", False)]
hab = df[df["espece"].isin(ESPECES) & df["profondeur"].notna()]
ordre_hab = hab.groupby("espece")["profondeur"].median().sort_values().index

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (colonne_var, titre, echelle_log) in zip(axes, variables):
    ax.boxplot([hab.loc[hab["espece"] == e, colonne_var].dropna() for e in ordre_hab], showfliers=False)
    ax.set_xticks(range(1, len(ordre_hab) + 1))
    ax.set_xticklabels(ordre_hab, rotation=25, ha="right")
    ax.set_title(titre)
    if echelle_log:
        ax.set_yscale("log")
plt.tight_layout()
plt.show()

hab.groupby("espece")[["profondeur", "pente", "dist_cote_km", "dist_talus_km"]].median().round(1).loc[ordre_hab]

In [ ]:
# Utilisation par rapport à l'ensemble des observations, par classe de profondeur
# Rapport > 1 : l'espèce est plus souvent dans cette classe que les autres espèces (elle-même exclue, pour ne pas se comparer à soi)
bornes = [0, 50, 100, 200, 500, 1000, 2000, 10_000]
noms = ["0-50", "50-100", "100-200", "200-500", "500-1000", "1000-2000", "plus de 2000"]
classe = pd.cut(hab["profondeur"], bornes, labels=noms)

rapports = {}
for espece in ordre_hab:
    ici = classe[hab["espece"] == espece].value_counts(normalize=True).reindex(noms).fillna(0)
    ailleurs = classe[hab["espece"] != espece].value_counts(normalize=True).reindex(noms).fillna(0)
    rapports[espece] = (ici + 0.01) / (ailleurs + 0.01)
rapports = pd.DataFrame(rapports).T

fig, ax = plt.subplots(figsize=(10, 3.8))
image = ax.imshow(np.log2(rapports.values), cmap="RdBu_r", vmin=-3, vmax=3, aspect="auto")
ax.set_xticks(range(len(noms)))
ax.set_xticklabels(noms)
ax.set_yticks(range(len(rapports)))
ax.set_yticklabels(rapports.index)
for i in range(rapports.shape[0]):
    for j in range(rapports.shape[1]):
        ax.text(j, i, f"{rapports.iloc[i, j]:.1f}", ha="center", va="center", fontsize=8)
ax.set_xlabel("classe de profondeur (m)")
plt.colorbar(image, label="log2 du rapport", shrink=0.8)
ax.set_title("Profondeur : l'espèce par rapport aux autres (rouge : plus fréquente que les autres)")
plt.tight_layout()
plt.show()

In [ ]:
# Jeunes et nouveau-nés selon la profondeur (dauphins)
dj = hab[hab["espece"].isin(["Grand dauphin", "Dauphin bleu et blanc"])].copy()
dj["classe"] = pd.cut(dj["profondeur"], bornes, labels=noms)
tab = dj.groupby("classe", observed=True).agg(
    reponses_jeunes=("pres_jeune", lambda s: s.isin(["oui", "non"]).sum()),
    jeunes_pct=("pres_jeune", lambda s: (s[s.isin(["oui", "non"])] == "oui").mean() * 100),
    reponses_nouveau_ne=("nouveau_ne", lambda s: s.notna().sum()),
    nouveau_ne_pct=("nouveau_ne", lambda s: (s.dropna() == "oui").mean() * 100)).round(0)
tab[tab["reponses_jeunes"] >= 15]

**À lire avec prudence** : une espèce « préfère » une profondeur seulement par rapport aux autres espèces observées avec le même effort. Un vrai **modèle de distribution** (type MaxEnt) demande l'effort réel ou un jeu de points de fond bien choisi, et un test de sensibilité : à envisager quand Miraceti fournira les traces d'effort.

## Pour aller plus loin
- **Effort réel** : demander les traces GPS à Miraceti. C'est ce qui permettrait de calculer un vrai taux de rencontre (observations par km), une probabilité de détection (section 15) et de savoir où l'on a cherché sans rien voir.
- **Pressions** : aires marines protégées, trafic maritime (AIS), ports, à croiser avec les mailles comme pour la bathymétrie.
- **Catalogue de photo-identification** : suivi d'individus, fidélité aux sites, capture-recapture.
- **Questions pour Miraceti** : signification de `type` (E, HE, O, A), unité de `distance`, doublons détectés en section 11.

## Annexe : déposer le fond de carte dans le bucket (une seule fois)
À lancer seulement par la personne qui a `fond_carte.gpkg` en local, dans `data/curated/observations/`.
Rien n'est écrasé : un fichier déjà dans le bucket est sauté.

In [ ]:
cle = FOND.relative_to("data").as_posix()
if not FOND.exists():
    print("absent en local :", FOND)
elif bucket.head(cle):
    print("déjà dans le bucket :", cle)
else:
    bucket.upload(FOND, cle)
    print("envoyé :", cle)